# RISE：主成分得分来自图像的哪些区域

RISE（Petsiuk et al., BMVC 2018）的黑盒取第 k 主成分的得分，加权求和前得分减去平均（见 `fmspace/rise.py`）。显著图：红 = 该区域把得分推高，蓝 = 推低。

先运行初始化格；其余各格只依赖它。每张图保存 `results/rise/<model>/<设定>/<image_id>.npz`：`sal`（显著图 `[K, H, W]`）、`scores`（各遮挡图的得分 `[N, K]`）、`s0`（原图得分）、`s_blank`（全遮得分）。

In [ ]:
import numpy as np
import pandas as pd
import torch
from PIL import Image

from fmspace.analysis import l2, pc_scorer, pca, rho
from fmspace.data import load_metadata
from fmspace.extract import load_features
from fmspace.models import load_model
from fmspace.paths import DATA_DIR, FEATURE_DIR, FIGURE_DIR, RISE_DIR
from fmspace.plots import saliency_figure
from fmspace.rise import rise

MODEL = "medsiglip"
COMPONENTS = {1: "eye_dr_level", 2: "eye"}  # 主成分 → 定方向用的标签
N_MASKS, GRID, P1, SEED = 4000, 7, 0.5, 0
PER_GRADE = 5

metadata = load_metadata(DATA_DIR).set_index("image_id", drop=False)
X = l2(load_features(FEATURE_DIR / f"{MODEL}.npz", metadata["image_id"]))
fitted = pca(X)
pcs = fitted.transform(X)[:, np.array(list(COMPONENTS)) - 1]
signs = np.array([np.sign(rho(pcs[:, j], metadata[label].rank())) for j, label in enumerate(COMPONENTS.values())])
scores = pd.DataFrame(pcs * signs, index=metadata.index, columns=[f"PC{k}" for k in COMPONENTS])
selected = scores.join(metadata[["eye", "eye_dr_level"]]).groupby("eye_dr_level").sample(PER_GRADE, random_state=SEED)

preprocess, forward = load_model(MODEL, "cuda")
model = pc_scorer(torch.autocast("cuda", dtype=torch.float16)(forward), fitted, list(COMPONENTS), signs)
OUT = RISE_DIR / MODEL / f"grid{GRID}_n{N_MASKS}_p{P1}"
OUT.mkdir(parents=True, exist_ok=True)

## 复现检查

`cosine`：重新提的特征与 `features/<model>.npz` 的余弦相似度，应接近 1。`PC1`、`PC2`：`model`（半精度）算出的得分与上面得分之差，应接近 0。

In [ ]:
ids = selected.index[:8]
with torch.inference_mode():
    x = preprocess([Image.open(metadata["path"][i]).convert("RGB") for i in ids])
    z = l2(forward(x).float().cpu().numpy())
    s = model(x).numpy()
check = pd.DataFrame({"cosine": (z * X[metadata.index.get_indexer(ids)]).sum(axis=1)}, index=ids)
check[scores.columns] = s - scores.loc[ids].to_numpy()
check

## 跑 RISE

每个 DR 等级 `PER_GRADE` 张。已有结果的图跳过。

In [ ]:
for image_id in selected.index:
    if (OUT / f"{image_id}.npz").exists():
        continue
    x = preprocess([Image.open(metadata["path"][image_id]).convert("RGB")])
    log = []

    def recorded(stack):
        log.append(model(stack))
        return log[-1]

    sal = rise(recorded, x, N=N_MASKS, s=GRID, p1=P1, seed=SEED)
    with torch.inference_mode():
        s0, s_blank = model(x)[0], model(torch.zeros_like(x))[0]
    np.savez(OUT / f"{image_id}.npz", sal=sal.numpy(), scores=torch.cat(log).numpy(), s0=s0.numpy(), s_blank=s_blank.numpy())

## 汇总

每张图、每个主成分一行：

- `s0`：原图得分；`s_mean`、`s_std`：遮挡图得分的平均与标准差；`s_blank`：全遮得分。
- `total`：显著图总和（按格子面积换算）；`ratio` = `total ÷ (s0 − s_mean)`，各区域作用可加时约为 1。
- `s_mean_additive` = `P1·s0 + (1 − P1)·s_blank`，可加时与 `s_mean` 相等。

In [ ]:
rows = []
for f in sorted(OUT.glob("*.npz")):
    d = np.load(f)
    h, w = d["sal"].shape[1:]
    cell = ((GRID + 1) * np.ceil(h / GRID) / GRID) * ((GRID + 1) * np.ceil(w / GRID) / GRID)
    for j, pc in enumerate(scores.columns):
        s0, s_mean, s_blank, total = d["s0"][j], d["scores"][:, j].mean(), d["s_blank"][j], d["sal"][j].sum() / cell
        rows.append({"image_id": f.stem, "dr": metadata["eye_dr_level"][f.stem], "eye": metadata["eye"][f.stem], "pc": pc,
                     "s0": s0, "s_mean": s_mean, "s_std": d["scores"][:, j].std(), "s_blank": s_blank,
                     "s_mean_additive": P1 * s0 + (1 - P1) * s_blank, "total": total, "ratio": total / (s0 - s_mean)})
summary = pd.DataFrame(rows).sort_values(["pc", "dr", "s0"])
summary.to_csv(OUT / "summary.csv", index=False)
summary

## 画图

按 DR 等级、PC1 得分排序；每列共用色标。

In [ ]:
done = selected.loc[[f.stem for f in OUT.glob("*.npz")]].sort_values(["eye_dr_level", "PC1"])
maps = [np.load(OUT / f"{i}.npz") for i in done.index]
images = [Image.open(metadata["path"][i]).convert("RGB") for i in done.index]
labels = [f"DR {r.eye_dr_level}, {r.eye}\n" + ", ".join(f"{c} {m['s0'][j]:+.2f}" for j, c in enumerate(scores.columns)) for r, m in zip(done.itertuples(), maps)]
fig = saliency_figure(images, [m["sal"] for m in maps], labels, list(scores.columns))
fig.savefig(FIGURE_DIR / f"rise_{MODEL}.png", dpi=300, bbox_inches="tight")